In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

In [2]:
pdfs = list(Path("./raw/").glob("*.pdf"))

In [3]:
import ollama
import fitz
import re


In [4]:
OLLAMA_URL = "http://localhost:11434"
TEXT_EMBED_MODEL = "nomic-embed-text"
LLM_MODEL = "llama3.2-vision"

In [ ]:
!ollama ls

In [5]:
from langchain_experimental.text_splitter import SemanticChunker
from langchain_ollama import OllamaEmbeddings
import concurrent
from langchain_community.document_loaders import PyPDFLoader

In [6]:
import json
import cv2
import os
import layoutparser as lp

In [7]:
EXTRACTED_DIR = "extracted_data"
IMAGES_DIR = os.path.join(EXTRACTED_DIR, "images")
os.makedirs(IMAGES_DIR, exist_ok=True)


In [8]:
FIGURE_LABELS = {"Picture", "Figure", "Image", "Fig"}
TABLE_LABELS  = {"Table"}
TARGET_LABELS = FIGURE_LABELS | TABLE_LABELS


In [9]:
def find_cap(page):
    x2,y1,x2,y2 = bbox 
    raw_blocks = page.get_text("blocks")
    best, min_dist = "" , float("inf")




    for b in raw_blocks:
        if b[6]!=0:
            continue
        tx0,ty0,tx1,ty1 = [c for c in b[:4]]
        text = b[4].replace("\n", " ").strip()

        horiz_overlap = max(0,min(x2,tx1) - max(x1,tx0))
        if horiz_overlap< 10 and (x2-x1 )>100:
            continue

        if box_type == "Figure":
            dist = ty0 - y2  # distance below the figure
            if -20 < dist < 400:
                score = dist - (200 if text.lower().startswith("fig") else 0)
                if score < min_dist:
                    min_dist, best = score, text


    return best

In [10]:
import os, json, fitz
import numpy as np
from pathlib import Path
from PIL import Image
#from surya.layout import LayoutPredictor
from tqdm.auto import tqdm
from surya.layout import LayoutPredictor, FoundationPredictor


In [ ]:

def main():
    pdfs = list(Path("raw/").glob("*.pdf"))
    if not pdfs:
        print(f"No PDFs found in '{raw}/'")
        return

    print(f"Found {len(pdfs)} PDF(s). Loading Surya layout model...")
    foundation = FoundationPredictor()       # loads base model (uses MPS on M2)
    predictor = LayoutPredictor(foundation)  # build layout predictor on top
    print("Model loaded.\n")

    all_metadata = []
    DPI = 150
    ZOOM = DPI / 72.0
    PAD = 20  # pixel padding around each crop

    for pdf_path in tqdm(pdfs, desc="Processing PDFs"):
        doc_name = pdf_path.stem
        print(f"\n→ {pdf_path.name}")

        try:
            doc = fitz.open(str(pdf_path))
        except Exception as e:
            print(f"  Failed to open: {e}")
            continue

        for page_idx in tqdm(range(len(doc)), desc=f"  Pages", leave=False):
            page = doc[page_idx]

            # Convert PDF page → PIL Image for Surya
            pix = page.get_pixmap(dpi=DPI)
            img = Image.frombytes("RGB", [pix.width, pix.height], pix.samples)

            # Run layout detection
            try:
                predictions = predictor([img])
            except Exception as e:
                print(f"  Skipping page {page_idx}: {e}")
                continue

            bboxes = predictions[0].bboxes

            figure_count = 0
            for box in bboxes:
                if box.label not in TARGET_LABELS:
                    continue

                box_type = "Table" if box.label in TABLE_LABELS else "Figure"

                # Apply padding, clamp to image bounds
                x1, y1, x2, y2 = [int(c) for c in box.bbox]
                x1 = max(0, x1 - PAD)
                y1 = max(0, y1 - PAD)
                x2 = min(img.width,  x2 + PAD)
                y2 = min(img.height, y2 + PAD)

                # Skip tiny detections (noise)
                if (x2 - x1) < 50 or (y2 - y1) < 50:
                    continue

                # Crop and save
                cropped = img.crop((x1, y1, x2, y2))
                out_name = f"{doc_name}_p{page_idx}_f{figure_count}.png"
                out_path = os.path.join(IMAGES_DIR, out_name)
                cropped.save(out_path)

                # Find caption from surrounding text
                caption = find_caption(page, (x1, y1, x2, y2), box_type, ZOOM)

                all_metadata.append({
                    "doc_name": doc_name,
                    "image_path": out_name,
                    "context": caption,
                    "page": page_idx,
                    "type": box_type
                })

                figure_count += 1

            if figure_count:
                print(f"  Page {page_idx}: {figure_count} figure(s) found")

        doc.close()

    # Save metadata
    metadata_path = os.path.join(EXTRACTED_DIR, "images_metadata.json")
    with open(metadata_path, "w", encoding="utf-8") as f:
        json.dump(all_metadata, f, indent=4)

    print(f"\n✅ Done! {len(all_metadata)} figures/tables extracted.")
    print(f"   Images → {IMAGES_DIR}/")
    print(f"   Metadata → {metadata_path}")


if __name__ == "__main__":
    main()


Found 18 PDF(s). Loading Surya layout model...
Model loaded.



Processing PDFs:   0%|          | 0/18 [00:00<?, ?it/s]


→ nature17151.pdf


  Pages:   0%|          | 0/5 [00:00<?, ?it/s]



Recognizing Layout: 100%|██████████| 1/1 [02:05<00:00, 125.44s/it]


Recognizing Layout: 100%|██████████| 1/1 [02:13<00:00, 133.90s/it]


Recognizing Layout: 100%|██████████| 1/1 [02:22<00:00, 142.53s/it]


Recognizing Layout: 100%|██████████| 1/1 [01:31<00:00, 91.84s/it]


Recognizing Layout: 100%|██████████| 1/1 [01:30<00:00, 90.39s/it]



→ nature12952.pdf


  Pages:   0%|          | 0/6 [00:00<?, ?it/s]



Recognizing Layout: 100%|██████████| 1/1 [01:46<00:00, 106.68s/it]


Recognizing Layout: 100%|██████████| 1/1 [01:39<00:00, 99.25s/it]



In [ ]:
embeddings = OllamaEmbeddings(
    model="mxbai-embed-large",
    base_url="http://localhost:11434"
)
text_splitter = SemanticChunker(
    embeddings, 
    breakpoint_threshold_type="percentile" 
)

In [ ]:
query = "Finding real-world clinical notes is famously difficult due to HIPAA and privacy regulations, but there are several high-quality open-ish repositories and synthetic alternatives that are standard in the NLP research community. As of 2026, here are the primary sources for clinical notes: The Gold Standards (Credentialed Access) These are real, de-identified clinical notes. While open source in spirit, they require a signed Data Use Agreement (DUA) and completion of a short ethics course (usually via CITI). MIMIC-IV (PhysioNet): The most widely used dataset in the world. It contains over 2 million de-identified clinical notes (discharge summaries, radiology reports, etc.) from the Beth Israel Deaconess Medical Center. Note: The MIMIC-IV-Note module was released as a separate extension to the core database."

In [ ]:
rec = text_splitter.split_text(query)

In [ ]:
len(rec)

In [ ]:
rec[0]

In [ ]:
from tqdm.auto import tqdm

In [ ]:
def semantic_chunking():
    embeddings = OllamaEmbeddings(
    model="qwen3-embedding",
    base_url="http://localhost:11434"
    )

    text_splitter = SemanticChunker(
    embeddings, 
    breakpoint_threshold_type="percentile" 
    )
    docs = []
    for doc in pdfs:
        loader = PyPDFLoader(doc)

        docs.extend(loader.load())
        
    semantic_chunks = []
    for doc1 in tqdm(docs, desc="Splitting documents"):
        semantic_chunks.extend(text_splitter.split_documents([doc1]))

    return semantic_chunks

semantic_chunking()

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
import concurrent.futures


In [ ]:

def load_single_pdf(pdf_path):
    loader = PyPDFLoader(pdf_path)
    return loader.load()

def chunk_single_doc(doc, text_splitter):
    return text_splitter.split_documents([doc])

def clean_text(text):
    # Collapse spaced-out characters: "W e  w i l l" -> "We will"
    text = re.sub(r'\b(\w) (\w) ', r'\1\2', text)  # iterative collapse
    # Run it multiple times since each pass only collapses pairs
    for _ in range(20):
        text = re.sub(r'(?<!\w)(\w) (\w)(?!\w)', r'\1\2', text)
    text = re.sub(r'\s+', ' ', text)
    return text.strip()


def semantic_chunking(pdfs):
    embeddings = OllamaEmbeddings(
        model="mxbai-embed-large",
        base_url="http://localhost:11434"
    )

    text_splitter = SemanticChunker(
        embeddings, 
        breakpoint_threshold_type="percentile" 
    )
    pre_splitter = RecursiveCharacterTextSplitter(
        chunk_size=500,   # characters, not tokens
        chunk_overlap=300
    )

    docs = []
    # 1. Parallelize PDF Loading
    # ThreadPool is great here because file I/O releases the GIL.
    print("Starting parallel PDF loading...")
    with concurrent.futures.ThreadPoolExecutor() as executor:
        # Map applies the function to the iterable concurrently
        results = list(tqdm(executor.map(load_single_pdf, pdfs), total=len(pdfs), desc="Loading PDFs"))
        for res in results:
            docs.extend(res)
    
    docs = pre_splitter.split_documents(docs)
        
    semantic_chunks = []
    
    MAX_WORKERS = 20
    
    print(f"Starting parallel chunking with {MAX_WORKERS} workers...")
    with concurrent.futures.ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
        # Submit tasks to the executor
        futures = [executor.submit(chunk_single_doc, doc, text_splitter) for doc in docs]
        
        # as_completed yields futures as they finish, keeping the progress bar accurate
        for future in tqdm(concurrent.futures.as_completed(futures), total=len(docs), desc="Splitting documents"):
            try:
                semantic_chunks.extend(future.result())
            except Exception as e:
                print(f"Skipping a chunk due to error: {e}")  # Don't crash on one bad doc

    final_chunks = []

    for chunk in semantic_chunks:
        if len(chunk.page_content) > 1800:
            final_chunks.extend(pre_splitter.split_documents([chunk]))
        else:
            final_chunks.append(chunk)
    
    for chun in final_chunks:
        chun.page_content = clean_text(chun.page_content)
    return final_chunks

semantic_chunks_parallel = semantic_chunking(pdfs)

In [ ]:
import chromadb
from chromadb.utils.embedding_functions import OllamaEmbeddingFunction
from langchain_ollama import OllamaEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter
from ollama import ResponseError

In [ ]:
client = chromadb.PersistentClient(path="chroma_db")

ef = OllamaEmbeddingFunction(
    model_name = "mxbai-embed-large",
    url="http://localhost:11434/api/embeddings"
)

In [ ]:
collection = client.get_or_create_collection(name="semantic_texts", embedding_function=ef)

In [ ]:

embeddings_model = OllamaEmbeddings(
    model="mxbai-embed-large", 
    base_url="http://localhost:11434"
    )

# Re-split any oversized chunks (more aggressive threshold)
safety_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500, 
    chunk_overlap=300)
    
final_chunks = []
for chunk in semantic_chunks_parallel:
    if len(chunk.page_content) > 1700:
        final_chunks.extend(safety_splitter.split_documents([chunk]))
    else:
        final_chunks.append(chunk)

print(f"Before: {len(semantic_chunks_parallel)}, After: {len(final_chunks)}")

texts = [doc.page_content for doc in final_chunks]
metadatas = [doc.metadata for doc in final_chunks]

# Embed one-by-one to avoid any batch issues
all_embeddings = []
for i in tqdm(range(len(texts)), desc="Embedding"):
    try:
        emb = embeddings_model.embed_documents([texts[i]])
        all_embeddings.extend(emb)
    except ResponseError:
        print(f"Chunk {i} too long ({len(texts[i])} chars), sub-splitting...")
        sub = safety_splitter.split_text(texts[i])
        # Just embed the first sub-chunk to keep 1:1 mapping
        all_embeddings.extend(embeddings_model.embed_documents([sub[0]]))

assert len(all_embeddings) == len(texts), "Mismatch!"
batch_size = 5000
for start in range(0, len(texts), batch_size):
    end = min(start + batch_size, len(texts))
    collection.add(
        ids=[f"chunk_{i}" for i in range(start, end)],
        documents=texts[start:end],
        metadatas=metadatas[start:end],
        embeddings=all_embeddings[start:end]
    )
    print(f"Added batch {start}-{end}")

In [ ]:
query = "What is the misfit function?"

#print(len(query))
embeddings_model = OllamaEmbeddings(
        model="mxbai-embed-large", 
        base_url="http://localhost:11434"
        )
safety_splitter = RecursiveCharacterTextSplitter(
        chunk_size=1700, 
        chunk_overlap=200
        )

query_to_chunks = []
if len(query) >1800:
    query_chunks = safety_splitter.split_documents(query)
    query_to_chunks.extend(query_chunks)
else:
    query_to_chunks.extend(query)


query_emb = embeddings_model.embed_documents(query_to_chunks)
    
dense_results = collection.query(query_embeddings= query_emb, n_results=5)
    
dense_ids = [int(id.split("_")[1]) for id in dense_results["ids"][0]]

for i in dense_ids:
    print(texts[i])
    #dic = metadatas[i]
    #print(dic)

In [ ]:
from rank_bm25 import BM25Okapi
import pickle

In [ ]:
tokenized_corpus = [doc.lower().split(" ") for doc in texts]
bm25 = BM25Okapi(tokenized_corpus)
with open("bm25_index.pkl", "wb") as f:
    pickle.dump(bm25, f)

In [ ]:
"""example"""

query = "universal conductance"
tokenized_query = query.lower().split(" ")
scores = bm25.get_scores(tokenized_query)
top_10 = np.argsort(scores)[::-1][:10]


In [ ]:
len(query)

In [ ]:
for i , inx in enumerate(top_10):
    if scores[inx] >12:
        print(scores[inx])
        dic = metadatas[inx]
        print(dic['source'])

In [ ]:
def hybrid_search(query, top_k, dense_weight=0.6, sparse_weight=0.4):
    k_cand = max(15, top_k * 3)

    # --- Sparse (BM25) retrieval ---
    tokenized_query = query.lower().split(" ")
    bm25_scores = bm25.get_scores(tokenized_query)
    sparse_top = np.argsort(bm25_scores)[::-1][:k_cand]

    # --- Dense (embedding) retrieval ---
    embeddings_model = OllamaEmbeddings(
        model="mxbai-embed-large", 
        base_url="http://localhost:11434"
    )

    # embed_query expects a single string, not a list
    query_emb = embeddings_model.embed_query(query)

    dense_results = collection.query(
        query_embeddings=[query_emb], 
        n_results=k_cand,
        include=["documents", "metadatas", "distances"]
    )

    dense_ids = [int(id.split("_")[1]) for id in dense_results["ids"][0]]
    # ChromaDB distances: lower = more similar (L2) → convert to similarity
    dense_dist = dense_results["distances"][0]
    max_dense = max(dense_dist) if dense_dist else 1.0
    dense_sim = {idx: 1 - (d / max_dense) for idx, d in zip(dense_ids, dense_dist)}

    # --- Normalise BM25 scores for the sparse candidates ---
    max_bm25 = bm25_scores[sparse_top[0]] if bm25_scores[sparse_top[0]] > 0 else 1.0
    sparse_sim = {int(idx): bm25_scores[idx] / max_bm25 for idx in sparse_top}

    # --- Weighted fusion ---
    all_candidates = set(dense_sim.keys()) | set(sparse_sim.keys())
    fused = {}
    for idx in all_candidates:
        d_score = dense_sim.get(idx, 0.0) * dense_weight
        s_score = sparse_sim.get(idx, 0.0) * sparse_weight
        fused[idx] = d_score + s_score

    # --- Rank and return top_k ---
    ranked = sorted(fused.items(), key=lambda x: x[1], reverse=True)[:top_k]

    results = []
    for idx, score in ranked:
        results.append({
            "chunk_index": idx,
            "text": texts[idx],
            "metadata": metadatas[idx],
            "hybrid_score": round(score, 4),
            "dense_score": round(dense_sim.get(idx, 0.0), 4),
            "sparse_score": round(sparse_sim.get(idx, 0.0), 4),
        })

    return results


In [ ]:
results = hybrid_search("universal conductance ﬂuctuations", top_k=5, dense_weight=0.5, sparse_weight=0.5)
for r in results:
    print(f"[{r['hybrid_score']}] {r['text']}...")


In [ ]:
import ollama

def rag_answer(query, top_k=10, dense_weight=0.6, sparse_weight=0.4):
    # 1. Retrieve context via hybrid search
    results = hybrid_search(query, top_k=top_k, 
                            dense_weight=dense_weight, 
                            sparse_weight=sparse_weight)

    # 2. Build context block from retrieved chunks
    context_block = ""
    for i, r in enumerate(results):
        source = r["metadata"].get("source", "unknown")
        context_block += f"--- Chunk {i+1} [source: {source}] ---\n"
        context_block += r["text"] + "\n\n"
    print(context_block)
    # 3. Prompt the LLM
    system_prompt = (
        "You are an expert scientific AI assistant. "
        "Answer the user's question using ONLY the provided context. "
        "Cite which chunk(s) you drew each part of the answer from. "
        "Generate a figure to describe the result where applicable"
        "look for mathematical descriptions"
        "If the context does not contain enough information, say so clearly."
    )

    user_prompt = (
        f"## Context\n{context_block}\n"
        f"## Question\n{query}\n\n"
        "Provide a detailed, well-structured answer."
    )

    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user",   "content": user_prompt},
    ]

    # 4. Stream the response
    print(f"Query: {query}\n{'='*60}")
    full_response = ""
    for chunk in ollama.chat(model="qwen3-vl:8b", messages=messages, stream=True):
        token = chunk["message"]["content"]
        full_response += token
        print(token, end="", flush=True)
    
    print(f"\n{'='*60}")
    return full_response, results


In [ ]:
answer, sources = rag_answer("What is the misfit function?")
